# Module 2: Sentiment / Emotion Classifier (LOCAL TRAINING, 4GB VRAM)
Dataset pulled via HuggingFace `datasets` API.

Approach: Fine-tuned DistilBERT, 6 emotions -> 3 buckets (negative/neutral/positive)

## IMPORTANT: settings tuned for FAST training on 4GB VRAM (~2 min)
- Uses 5000 training samples (subset) for fast training
- batch_size=16 with fp16 -- fits 4GB VRAM, fast processing
- gradient_accumulation_steps=2 -- effective batch size = 16*2 = 32
- 1 epoch only with higher learning rate (5e-5) for fast convergence
If you still hit CUDA OOM, drop per_device_train_batch_size to 8 and raise
gradient_accumulation_steps to 4.

This runs in ~2 minutes on a 4GB GPU. For full training (better accuracy),
increase epochs to 3 and remove the sample size limit.

In [1]:
# Dependencies: pip install transformers datasets accelerate scikit-learn torch (with CUDA build)
# For GPU (CUDA 12.6): pip install torch --index-url https://download.pytorch.org/whl/cu126

import numpy as np
import pandas as pd
import torch
import json
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

CUDA available: True
GPU: Quadro T2000
VRAM (GB): 3.9


## Load dataset, remap 6 emotions -> 3 buckets

In [2]:
ds = load_dataset("dair-ai/emotion")

EMOTION_NAMES = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
BUCKET_MAP = {
    "sadness": "negative", "anger": "negative", "fear": "negative",
    "surprise": "neutral", "joy": "positive", "love": "positive",
}
BUCKET_NAMES = ["negative", "neutral", "positive"]
BUCKET_TO_ID = {name: i for i, name in enumerate(BUCKET_NAMES)}

def remap_labels(example):
    example["bucket_label"] = BUCKET_TO_ID[BUCKET_MAP[EMOTION_NAMES[example["label"]]]]
    return example

ds = ds.map(remap_labels)

# Use subset of 5000 training samples for fast ~2min training
TRAIN_SAMPLES = 5000
ds["train"] = ds["train"].shuffle(seed=42).select(range(min(TRAIN_SAMPLES, len(ds["train"]))))

print(ds["train"].to_pandas()["bucket_label"].value_counts())

bucket_label
0    2762
2    2069
1     169
Name: count, dtype: int64


## Tokenize

In [3]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64)

ds_tok = ds.map(tokenize_fn, batched=True)
ds_tok = ds_tok.remove_columns(["text", "label"]).rename_column("bucket_label", "labels")
ds_tok.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Load model

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={i: n for i, n in enumerate(BUCKET_NAMES)}, label2id=BUCKET_TO_ID,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Metrics

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

## Train (4GB-VRAM-safe settings)

In [6]:
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="./sentiment_ckpt",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,                  # higher LR for fast 1-epoch convergence
    per_device_train_batch_size=16,      # fits 4GB VRAM with fp16
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,       # effective batch size = 16*2 = 32
    num_train_epochs=1,                  # 1 epoch for ~2min training
    weight_decay=0.01,
    fp16=use_fp16,                       # mixed precision saves ~40% VRAM
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=25,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=ds_tok["train"], eval_dataset=ds_tok["validation"],
    processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.397525,0.214244,0.928000,0.661536


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=157, training_loss=0.7945691795106147, metrics={'train_runtime': 209.5913, 'train_samples_per_second': 23.856, 'train_steps_per_second': 0.749, 'total_flos': 60927810710976.0, 'train_loss': 0.7945691795106147, 'epoch': 1.0})

## Evaluate

In [7]:
test_results = trainer.evaluate(ds_tok["test"])
print(test_results)

preds_output = trainer.predict(ds_tok["test"])
preds = np.argmax(preds_output.predictions, axis=-1)
print(classification_report(preds_output.label_ids, preds, target_names=BUCKET_NAMES))

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.397525,0.196118,1,0.937000,0.672059


{'eval_loss': 0.19611825048923492, 'eval_accuracy': 0.937, 'eval_f1_macro': 0.6720594867800683}


              precision    recall  f1-score   support

    negative       0.94      0.97      0.96      1080
     neutral       1.00      0.06      0.11        66
    positive       0.93      0.96      0.95       854

    accuracy                           0.94      2000
   macro avg       0.96      0.66      0.67      2000
weighted avg       0.94      0.94      0.92      2000



## Qualitative check on customer-support-style text (domain shift sanity check)

In [8]:
support_style_samples = [
    ("This is the third time I've contacted you about my missing order. Absolutely ridiculous.", "negative"),
    ("Can you tell me the status of my order #4521?", "neutral"),
    ("Thanks so much for sorting that out so quickly, really appreciate it!", "positive"),
    ("I want a refund NOW, this is unacceptable service.", "negative"),
    ("Hi, I'd like to update my shipping address please.", "neutral"),
]

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for text, expected in support_style_samples:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_name = BUCKET_NAMES[torch.argmax(logits, dim=-1).item()]
    match = "OK" if pred_name == expected else "MISMATCH"
    print(f"[{match}] expected={expected:9s} predicted={pred_name:9s} | {text}")

[OK] expected=negative  predicted=negative  | This is the third time I've contacted you about my missing order. Absolutely ridiculous.
[MISMATCH] expected=neutral   predicted=positive  | Can you tell me the status of my order #4521?
[OK] expected=positive  predicted=positive  | Thanks so much for sorting that out so quickly, really appreciate it!
[OK] expected=negative  predicted=negative  | I want a refund NOW, this is unacceptable service.
[MISMATCH] expected=neutral   predicted=positive  | Hi, I'd like to update my shipping address please.


## Save model directly into local_app/models/sentiment/

In [9]:
OUTPUT_DIR = "../local_app/models/sentiment"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(os.path.join(OUTPUT_DIR, "bucket_labels.json"), "w") as f:
    json.dump({str(i): n for i, n in enumerate(BUCKET_NAMES)}, f, indent=2)

print("Saved to", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ../local_app/models/sentiment
['.gitkeep', 'config.json', 'tokenizer_config.json', 'tokenizer.json', 'bucket_labels.json', 'model.safetensors']
